<a href="https://colab.research.google.com/github/TurkuNLP/intro-to-nlp/blob/master/course_project_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to HLT Project (from template)

- Student(s) Name(s): Lauris Seebeck
- Date: 23.4.2026-
- Chosen Corpus: IMDB
- Contributions (if group project): None

### Corpus information

- Description of the chosen corpus:
- Paper(s) and other published materials related to the corpus:
- State-of-the-art performance (best published results) on this corpus:

---

## 1. Setup

In [1]:
import datasets
import evaluate
import transformers
import torch
import numpy as np

from pprint import pprint

---

## 2. Data download and preprocessing

### 2.1. Download the corpus

In [2]:
# Loading the dataset

dset = datasets.load_dataset('imdb')
dset=dset.shuffle()
del dset["unsupervised"]
pprint(dset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})


### 2.2. Preprocessing

In [3]:
base_tokenizer = transformers.AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
tokenizer = base_tokenizer.train_new_from_iterator(
    dset["train"]["text"],
    vocab_size=15000
)

collator=transformers.DataCollatorWithPadding(tokenizer)
small_data=[tokenizer("Hi there"), tokenizer("A little longer text!")]
print("small_data:\n")
pprint(small_data)
print("\n\ncollated:\n")
small_batch=collator(small_data)
pprint(small_batch)




small_data:

[{'attention_mask': [1, 1, 1, 1],
  'input_ids': [2, 6004, 310, 3],
  'token_type_ids': [0, 0, 0, 0]},
 {'attention_mask': [1, 1, 1, 1, 1, 1, 1],
  'input_ids': [2, 43, 566, 3056, 4274, 5, 3],
  'token_type_ids': [0, 0, 0, 0, 0, 0, 0]}]


collated:

{'attention_mask': tensor([[1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1]]),
 'input_ids': tensor([[   2, 6004,  310,    3,    0,    0,    0],
        [   2,   43,  566, 3056, 4274,    5,    3]]),
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0]])}


In [4]:
def encode(examples):
    return tokenizer(examples['text'],
                     #truncation=True,
                     #max_length=256
                     )

dset_tokenized = dset.map(encode,batched=True,num_proc=4)

for key,val in dset_tokenized["train"][0].items():
    print(key,":",val)

Map (num_proc=4):   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (717 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (736 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (545 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (846 > 512). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (669 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (686 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (535 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1347 > 512). Running this sequence through the model will result in indexing errors


text : This movie is a ripoff of James Cain's novel, THE POSTMAN ALWAYS RINGS TWICE. Apparently, the director and producer never bothered to pay for rights to this story--perhaps the fact that we were in the middle of fighting the Italians in WWII might account for their forgetting to consider royalties! Despite this, the movie isn't really just an Italian version of the Hollywood movie. In some ways it's a lot better and in other ways, it is definitely not.<br /><br />The three central characters in this movie are really pretty ugly people. In fact, the male and female lovers are a bit icky-looking. The male lead is pretty ordinary except for his profuse body hair (particularly on the back and shoulders) and his lady love is, to put it frankly, unattractive. They are a very, very far cry from Lana Turner and John Garfield in the Hollywood version. And the ill-fated husband is really, really obese and loves to walk around shirtless--and his counterpart in the American film, Cecil Kella

---

## 3. Machine learning model

### 3.1. Model training

In [5]:
# A Transformers library model wants a config,
# I can simply inherit from the base
# class for pretrained configs
# We don't really need to do anything very special
class MLPConfig(transformers.PretrainedConfig):
    pass

# This is the model
class MLP(transformers.PreTrainedModel):

    config_class=MLPConfig

    # In the initialization method, one instantiates the layers
    # these will be, for the most part the trained parameters of the model
    def __init__(self,config):
        super().__init__(config)
        self.all_tied_weights_keys = {} #Annoying bug in Transformers, must have this here or else we crash on saved model load
        #### HERE WE CREATE THE MODEL'S LAYERS:
        self.vocab_size=config.vocab_size #embedding matrix row count
        # Build and initialize embedding of vocab size x hidden size
        assert tokenizer.vocab["[PAD]"]==0 #let's make sure our assumption of pad==0 holds!

        self.embedding=torch.nn.Embedding(num_embeddings=self.vocab_size,embedding_dim=config.hidden_size,padding_idx=0)
        # Initialize the embeddings to random values
        # Note! This function is relatively clever and keeps the embedding for 0, the padding, pure zeros
        torch.nn.init.uniform_(self.embedding.weight.data,-0.001,0.001) #initialize the embeddings with small random values

        # This takes care of the lower half of the network, now the upper half
        # Output layer: hidden size x output size
        self.output=torch.nn.Linear(in_features=config.hidden_size,out_features=config.nlabels)
        # Now we have the parameters of the model
        self.loss=torch.nn.CrossEntropyLoss() #This loss is meant for classification, so let's use it


    # The computation of the model is put into the forward() function
    # it receives a batch of data and optionally the correct `labels`
    #
    # If given `labels` we should return (loss,output)
    # if not, then we should return (output,)
    # that way the model can be used both for training and for inference
    def forward(self,input_ids,labels=None,**kwargs):
        #1) sum up the embeddings of the items
        embedded=self.embedding(input_ids) #(batch,ids)->(batch,ids,embedding_dim)
        # Since the Embedding keeps the first row of the matrix pure zeros, we don't need to worry about the padding 0 index
        # so next we sum the embeddings across the word dimension
        # (batch,ids,embedding_dim) -> (batch,embedding_dim)
        embedded_summed=torch.sum(embedded,dim=1)

        #2) apply non-linearity
        # (batch,embedding_dim) -> (batch,embedding_dim)
        projected=torch.tanh(embedded_summed) #Note how non-linearity is applied here and not when configuring the layer in __init__()

        #3) and now apply the upper, output layer of the network
        # (batch,embedding_dim) -> (batch, num_of_classes i.e. 2 in our case)
        logits=self.output(projected)

        # ...and that's all there is to it!

        #print("input_ids.shape",input_ids.shape)
        #print("embedded.shape",embedded.shape)
        #print("embedded_summed.shape",embedded_summed.shape)
        #print("projected.shape",projected.shape)
        #print("logits.shape",logits.shape)

        # If we have labels, we ought to calculate the loss
        if labels is not None:
            # You run the loss as loss(model_output,correct_labels)
            return (self.loss(logits,labels),logits) #2-tuple, i.e. pair of values returned
        else:
            # No labels, so just return the logits
            return (logits,) #this weird syntax means 1-tuple



In [6]:
# Configure the model:
#   these parameters are used in the model's __init__()
mlp_config=MLPConfig(vocab_size=tokenizer.vocab_size,hidden_size=20,nlabels=2)
print("mlp config:", mlp_config)

# And now we can instantiate it
mlp=MLP(mlp_config)
print("mlp",mlp)
#we can make a little test with the small test batch we made earlier
#since it has no true labels, it should return a 1-tuple, which it will
out=mlp(input_ids=small_batch["input_ids"])
print("Output on one batch:",out)

mlp config: MLPConfig {
  "hidden_size": 20,
  "nlabels": 2,
  "transformers_version": "5.3.0",
  "vocab_size": 15000
}

mlp MLP(
  (embedding): Embedding(15000, 20, padding_idx=0)
  (output): Linear(in_features=20, out_features=2, bias=True)
  (loss): CrossEntropyLoss()
)
Output on one batch: (tensor([[-0.1812,  0.0375],
        [-0.1809,  0.0371]], grad_fn=<AddmmBackward0>),)


### 3.2 Hyperparameter optimization

In [7]:
trainer_args = transformers.TrainingArguments(
    "mlp_checkpoints", #save checkpoints here
    eval_strategy="steps", #...and not epochs (step is "one batch", epoch is "one full pass through the whole data")
    logging_strategy="steps",
    eval_steps=500, #eval every 500 steps
    logging_steps=500,
    learning_rate=3e-5, #learning rate of the gradient descent
    max_steps=12000,
    load_best_model_at_end=True, #when done, load the best model you have (which is not necessarily the one after the last step)
    per_device_train_batch_size=16 #batch size
)

pprint(trainer_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=500,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False

#### Hyperparameter turning scores:


    eval_steps=500, #eval every 500 steps
    logging_steps=500,
    learning_rate=5e-5,
    max_steps=10000,
    per_device_train_batch_size=16 
Evaluation: 0.207048	0.275306	0.894000

    eval_steps=500, 
    logging_steps=500,
    learning_rate=5e-5,
    max_steps=10000,
    per_device_train_batch_size=8

Evaluation: 0.239539	0.282307	0.892000

    eval_steps=500, 
    logging_steps=500,
    learning_rate=4e-5,
    max_steps=15000,
    per_device_train_batch_size=8

Evaluation: 0.219509	0.309143	0.871000

    eval_steps=500, 
    logging_steps=500,
    learning_rate=4e-5,
    max_steps=15000,
    per_device_train_batch_size=16

Evaluation: 0.188986	0.281887	0.890000

    eval_steps=500, 
    logging_steps=500,
    learning_rate=3e-5,
    max_steps=15000,
    per_device_train_batch_size=16

Evaluation: 0.218091	0.272472	0.901000

### 3.3. Evaluation on test set

In [8]:
accuracy = evaluate.load("accuracy")

def compute_accuracy(outputs_and_labels):
    outputs, labels = outputs_and_labels
    predictions = np.argmax(outputs, axis=-1) #pick the index of the "winning" label among the outputs, i.e. argmax
    return accuracy.compute(predictions=predictions, references=labels)

In [9]:
# Make a new model, this will also initialize it
mlp = MLP(mlp_config)


# Argument gives the number of evaluation tries of patience before early stopping
# i.e. training is stopped when the evaluation loss fails to improve
# certain number of times the model is evaluated, in this case 5 consecutive times
early_stopping = transformers.EarlyStoppingCallback(5)

trainer = transformers.Trainer(
    model=mlp,
    args=trainer_args,
    train_dataset=dset_tokenized["train"],
    eval_dataset=dset_tokenized["test"].select(range(1000)), #make a smaller subset to evaluate on
    compute_metrics=compute_accuracy,
    data_collator=collator,
    callbacks=[early_stopping]
)

trainer.train()

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
500,0.652438,0.612627,0.777000
1000,0.584510,0.548846,0.814000
1500,0.527049,0.499059,0.840000
2000,0.465780,0.457096,0.843000
2500,0.435015,0.421842,0.859000
3000,0.405441,0.397523,0.866000
3500,0.368400,0.374035,0.872000
4000,0.355189,0.358610,0.875000
4500,0.338776,0.344175,0.877000
5000,0.322232,0.333124,0.878000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12000, training_loss=0.34157341448465983, metrics={'train_runtime': 70.1974, 'train_samples_per_second': 2735.145, 'train_steps_per_second': 170.947, 'total_flos': 43384370400.0, 'train_loss': 0.34157341448465983, 'epoch': 7.677543186180422})

---

## 4. Results and summary

### 4.1 Corpus insights

(Briefly discuss what you learned about the corpus and its annotation)

### 4.2 Results

(Briefly summarize your results)

### 4.3 Relation to state of the art

(Compare your results to the state-of-the-art performance)

---

## 5. Bonus Task (optional)

### 5.1. Annotating out-of-domain documents

(Briefly describe the chosen out-of-domain documents)

(Briefly describe the process of annotation)

### 5.2 Conversion into dataset

In [10]:
# Your code to convert the annotations into a dataset here

### 5.3. Model evaluation on out-of-domain test set

In [11]:
# Your code to evaluate the model on the out-of-domain test set here

### 5.4 Bonus task results

(Present the results of the evaluation on the out-of-domain test set)

### 5.5. Annotated data

In [12]:
# Include your annotated out-of-domain data here